# NIFTY100 Financial Analytics

## Sprint 2 – Day 8

# Risk Analytics

This notebook evaluates the financial risk of NIFTY100 companies.

### Objectives

- Connect SQLite Database
- Calculate Volatility
- Calculate Sharpe Ratio
- Calculate Beta
- Calculate Maximum Drawdown
- Build Risk Scorecard
- Export Results

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path("../")

DATABASE = ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(DATABASE)

print("Connected Successfully!")

Connected Successfully!


In [3]:
profit = pd.read_sql(
    "SELECT * FROM profitandloss",
    conn
)

companies = pd.read_sql(
    "SELECT * FROM companies",
    conn
)

In [4]:
print(profit.shape)
print(companies.shape)

(1276, 15)
(92, 12)


## Volatility Analysis

Volatility measures how much company sales fluctuate over time.

Higher standard deviation = Higher Risk.

In [5]:
volatility = (
    profit
    .groupby("company_id")["sales"]
    .std()
    .reset_index()
)

volatility.columns = [
    "company_id",
    "sales_volatility"
]

volatility = volatility.sort_values(
    by="sales_volatility",
    ascending=False
)

volatility.head(10)

,company_id,sales_volatility
75,RELIANCE,231204.319350
69,ONGC,186806.787721
49,IOC,166741.787157
58,LICI,116169.281942
20,BPCL,97800.287085
77,SBIN,92049.492227
38,HDFCBANK,90262.061411
83,TATAMOTORS,73529.015780
86,TCS,61307.824157
41,HINDALCO,53931.111312


In [6]:
OUTPUT = ROOT / "outputs"

OUTPUT.mkdir(exist_ok=True)

volatility.to_csv(
    OUTPUT / "volatility_analysis.csv",
    index=False
)

print("Volatility Analysis Saved!")

Volatility Analysis Saved!


## Sharpe Ratio

Sharpe Ratio measures return relative to volatility.

Higher Sharpe Ratio indicates better risk-adjusted performance.

In [7]:
returns = profit[
    [
        "company_id",
        "year",
        "net_profit"
    ]
].copy()

returns = returns.sort_values(
    ["company_id", "year"]
)

returns["previous_profit"] = (
    returns
    .groupby("company_id")["net_profit"]
    .shift(1)
)

returns["return_pct"] = (
    (
        returns["net_profit"] -
        returns["previous_profit"]
    )
    /
    returns["previous_profit"]
) * 100

In [8]:
sharpe = (
    returns
    .groupby("company_id")["return_pct"]
    .agg(
        mean_return="mean",
        volatility="std"
    )
    .reset_index()
)

sharpe["sharpe_ratio"] = (
    sharpe["mean_return"] /
    sharpe["volatility"]
)

sharpe = sharpe.replace(
    [np.inf, -np.inf],
    np.nan
)

sharpe = sharpe.dropna()

sharpe.sort_values(
    "sharpe_ratio",
    ascending=False,
    inplace=True
)

sharpe.head(10)

,company_id,mean_return,volatility,sharpe_ratio
38,HDFCBANK,21.798053,7.399250,2.945982
57,KOTAKBANK,21.437384,7.804234,2.746891
0,ABB,20.452466,11.877480,1.721953
48,INFY,9.525882,5.567814,1.710884
23,CHOLAFIN,24.284782,14.398299,1.686642
76,SBILIFE,12.052960,7.383554,1.632406
12,BAJAJFINSV,18.831058,12.634048,1.490501
14,BAJFINANCE,35.252630,25.111602,1.403838
42,HINDUNILVR,9.178889,7.030503,1.305581
29,DMART,35.126153,27.375718,1.283114


In [9]:
sharpe.to_csv(
    OUTPUT / "sharpe_ratio.csv",
    index=False
)

print("Sharpe Ratio Saved!")

Sharpe Ratio Saved!


## Beta Analysis (Simplified)

Beta measures how sensitive a company's performance is compared to the market.

In this project, Year-over-Year Sales Growth is used as a proxy for market returns for demonstration purposes.

In [10]:
beta_data = profit[
    [
        "company_id",
        "year",
        "sales"
    ]
].copy()

beta_data = beta_data.sort_values(
    ["company_id", "year"]
)

# Calculate yearly sales growth
beta_data["sales_growth"] = (
    beta_data
    .groupby("company_id")["sales"]
    .pct_change()
)

# Calculate standard deviation of sales growth
beta_analysis = (
    beta_data
    .groupby("company_id")["sales_growth"]
    .std()
    .reset_index()
)

beta_analysis.rename(
    columns={"sales_growth": "beta_proxy"},
    inplace=True
)

beta_analysis = beta_analysis.sort_values(
    by="beta_proxy",
    ascending=False
)

beta_analysis.head(10)

,company_id,beta_proxy
54,JIOFIN,28.398332
96,VEDL,6.635460
45,ICICIPRULI,0.936610
13,BAJAJHLDNG,0.693419
98,ZOMATO,0.625981
3,ADANIGREEN,0.593924
50,IRCTC,0.551029
39,HDFCLIFE,0.484093
87,TECHM,0.478164
69,ONGC,0.422247


In [11]:
beta_analysis.to_csv(
    OUTPUT / "beta_analysis.csv",
    index=False
)

print("Beta Analysis Saved Successfully!")

Beta Analysis Saved Successfully!


## Maximum Drawdown

Maximum Drawdown measures the largest decline from a company's historical peak sales.

In [12]:
drawdown_list = []

for company in profit["company_id"].unique():

    df = (
        profit[profit["company_id"] == company]
        .sort_values("year")
    )

    df = df.dropna(subset=["sales"])

    if len(df) < 2:
        continue

    peak = df["sales"].cummax()

    drawdown = (df["sales"] - peak) / peak

    drawdown_list.append({
        "company_id": company,
        "max_drawdown_pct": round(drawdown.min() * 100, 2)
    })

max_drawdown = pd.DataFrame(drawdown_list)

max_drawdown = max_drawdown.sort_values(
    by="max_drawdown_pct"
)

max_drawdown.head(10)

,company_id,max_drawdown_pct
49,IRCTC,-65.68
18,BHEL,-64.61
99,INDIGO,-59.05
58,LODHA,-56.21
13,BAJAJHLDNG,-50.12
45,ICICIPRULI,-49.29
2,ADANIENT,-47.25
28,DLF,-45.46
74,RELIANCE,-37.12
39,HDFCLIFE,-33.34


In [13]:
max_drawdown.to_csv(
    OUTPUT / "max_drawdown.csv",
    index=False
)

print("Maximum Drawdown Saved Successfully!")

Maximum Drawdown Saved Successfully!


## Risk Scorecard

This section combines all calculated risk metrics into a single risk score.

Metrics Used:
- Sales Volatility
- Sharpe Ratio
- Beta Proxy
- Maximum Drawdown

Lower Risk Score = Lower Risk Company
Higher Risk Score = Higher Risk Company

In [14]:
# Merge all risk metrics

risk_scorecard = volatility.merge(
    sharpe[
        [
            "company_id",
            "sharpe_ratio"
        ]
    ],
    on="company_id",
    how="outer"
)

risk_scorecard = risk_scorecard.merge(
    beta_analysis,
    on="company_id",
    how="outer"
)

risk_scorecard = risk_scorecard.merge(
    max_drawdown,
    on="company_id",
    how="outer"
)

# Replace missing values

risk_scorecard.fillna(0, inplace=True)

risk_scorecard.head()

,company_id,sales_volatility,sharpe_ratio,beta_proxy,max_drawdown_pct
0,ABB,1443.650892,1.721953,0.092111,0.00
1,ADANIENSOL,6925.362887,0.000000,0.000000,-13.05
2,ADANIENT,30102.001427,0.421122,0.366702,-47.25
3,ADANIGREEN,3693.593609,0.213970,0.593924,0.00
4,ADANIPORTS,7856.560210,0.544381,0.133685,-3.51


In [15]:
# Calculate Risk Score

risk_scorecard["risk_score"] = (

    risk_scorecard["sales_volatility"] * 0.40

    +

    risk_scorecard["beta_proxy"] * 100 * 0.20

    +

    abs(risk_scorecard["max_drawdown_pct"]) * 0.30

    -

    risk_scorecard["sharpe_ratio"] * 10 * 0.10

)

risk_scorecard = risk_scorecard.sort_values(
    by="risk_score",
    ascending=True
)

risk_scorecard.head(20)

,company_id,sales_volatility,sharpe_ratio,beta_proxy,max_drawdown_pct,risk_score
13,BAJAJHLDNG,477.330168,0.998246,0.693419,-50.12,218.838203
65,NAUKRI,748.660231,-0.520856,0.168257,-14.02,307.556094
94,UNITDSPR,1134.953557,0.325782,0.094094,-22.62,462.323515
67,NHPC,1175.431296,0.175321,0.098320,-10.07,474.984592
50,IRCTC,1280.608002,0.520018,0.551029,-65.68,542.447765
55,JSWENERGY,1338.756426,0.317358,0.132989,-29.54,546.707000
28,DLF,1333.621626,0.110057,0.175228,-45.46,550.481144
0,ABB,1443.650892,1.721953,0.092111,0.00,577.580626
9,ATGL,1444.853521,0.797189,0.303145,-9.55,586.072127
26,DABUR,2090.336183,0.914387,0.058184,-2.32,837.079762


In [16]:
risk_scorecard.to_csv(
    OUTPUT / "risk_scorecard.csv",
    index=False
)

print("Risk Scorecard Saved Successfully!")

Risk Scorecard Saved Successfully!


## Top Low Risk Companies

These are the companies with the lowest calculated risk scores.

In [17]:
top_low_risk = risk_scorecard.sort_values(
    by="risk_score",
    ascending=True
).head(20)

top_low_risk

,company_id,sales_volatility,sharpe_ratio,beta_proxy,max_drawdown_pct,risk_score
13,BAJAJHLDNG,477.330168,0.998246,0.693419,-50.12,218.838203
65,NAUKRI,748.660231,-0.520856,0.168257,-14.02,307.556094
94,UNITDSPR,1134.953557,0.325782,0.094094,-22.62,462.323515
67,NHPC,1175.431296,0.175321,0.098320,-10.07,474.984592
50,IRCTC,1280.608002,0.520018,0.551029,-65.68,542.447765
55,JSWENERGY,1338.756426,0.317358,0.132989,-29.54,546.707000
28,DLF,1333.621626,0.110057,0.175228,-45.46,550.481144
0,ABB,1443.650892,1.721953,0.092111,0.00,577.580626
9,ATGL,1444.853521,0.797189,0.303145,-9.55,586.072127
26,DABUR,2090.336183,0.914387,0.058184,-2.32,837.079762


In [18]:
top_low_risk.to_csv(
    OUTPUT / "top_low_risk.csv",
    index=False
)

print("Top Low Risk Companies Saved Successfully!")

Top Low Risk Companies Saved Successfully!


## Top High Risk Companies

These are the companies with the highest calculated risk scores.

In [19]:
top_high_risk = risk_scorecard.sort_values(
    by="risk_score",
    ascending=False
).head(20)

top_high_risk

,company_id,sales_volatility,sharpe_ratio,beta_proxy,max_drawdown_pct,risk_score
75,RELIANCE,231204.319350,1.054403,0.242127,-37.12,92496.651887
69,ONGC,186806.787721,0.325220,0.422247,-28.76,74739.462815
49,IOC,166741.787157,-0.203599,0.260406,-31.09,66711.453583
58,LICI,116169.281942,0.450804,0.021255,0.00,46467.687067
20,BPCL,97800.287085,0.363686,0.221402,-28.97,39132.870186
77,SBIN,92049.492227,0.315534,0.074580,-0.64,36821.164947
38,HDFCBANK,90262.061411,2.945982,0.155533,0.00,36104.989240
83,TATAMOTORS,73529.015780,0.118110,0.123444,-17.27,29419.138076
86,TCS,61307.824157,1.015234,0.078035,0.00,24523.675128
41,HINDALCO,53931.111312,-0.168127,0.149724,-9.50,21578.457140


In [20]:
top_high_risk.to_csv(
    OUTPUT / "top_high_risk.csv",
    index=False
)

print("Top High Risk Companies Saved Successfully!")

Top High Risk Companies Saved Successfully!


In [21]:
conn.close()

print("SQLite Connection Closed Successfully!")

SQLite Connection Closed Successfully!
